# Session 24 · Embeddings — searching by meaning
**Homework notebook.** In class we drew it, saw it live and ran it on the projector. Here you run every step yourself,
then write your own semantic search in `s24/search.py` (four TODOs) and prove it with the checker.

| Part | What | Time |
|---|---|---|
| A | Why this matters, the words you need, and how today fits the course | 15 min reading |
| B | Run it: keyword search → your first embedding → cosine → meaning search → measure it → compare two models | 30 min |
| C | Your homework file: four TODOs, run, check, commit | 45 min |
| D | Interview questions and what comes next | 10 min |

**Before you start** (course repo folder, PowerShell):
```powershell
git pull
uv sync                                   # NEW today: fastembed (embedding models on your CPU)
uv run jupyter lab module04/s24_embeddings.ipynb
```
The first time you embed anything, the model (**BAAI/bge-small-en-v1.5**, about 67 MB) downloads once. Use home Wi-Fi or a hotspot
if an office network blocks it. After that, everything here runs **offline and free**: no API key, no tokens.

---
# Part A — Why this matters

### The problem you've already met
In Mini-Project 1, ShopAssist knew **seven facts** — the ones in its system prompt. Ask it about warranty, cash on delivery
or a coupon and it either says "I don't know" or, worse, guesses. A real store has a help centre with hundreds of answers.
You can't paste them all into the prompt: it would cost a fortune in tokens, be slow, and the model gets *worse* at
finding the right line in a huge prompt.

So we need to **find the right paragraph first**, then give only that paragraph to the model. That's the "R" in RAG
(Retrieval-Augmented Generation, Module 5). Today is the first half: **finding**.

### Why Ctrl+F isn't enough
Customers don't use the help centre's words.

| Customer types | The article that answers it says |
|---|---|
| "my earbuds stopped working" | "Warranty periods … headphones, earbuds … defects" |
| "the parcel arrived smashed" | "Damaged, defective or wrong items" |
| "when will I get my money back" | "Refund timelines" |

"Smashed" and "damaged" share no letters. A keyword search scores the second row **zero**. We need search that understands
that *smashed ≈ damaged*, *money back ≈ refund*. That's what an **embedding** gives us.

### Where this goes in industry
Every "chat with your documents" product — bank policy assistants, insurance claim helpers, hospital SOP bots, internal
IT helpdesks — starts with exactly today's step. ShopAssist Pro (your take-home kit) does this in `app/rag/`.

### The words you need (full list: `module04/terms_s24.md`)

| Term | Plain meaning | Analogy |
|---|---|---|
| **Embedding** | A list of numbers (a vector) that represents the *meaning* of a text | GPS coordinates for meaning |
| **Embedding model** | The model that turns text into that list. Not a chat model — it doesn't write, it only maps | A translator into "number language" |
| **Dimensions** | How many numbers are in the list. bge-small: 384. Bigger often = finer meaning, more storage | Map with 2 coordinates vs 384 |
| **Dense vector** | Every number is used (no zeros) — captures meaning | A fingerprint |
| **Sparse vector** | Mostly zeros, one slot per word — captures spelling (keyword search, BM25) | A word checklist |
| **Cosine similarity** | How much two vectors point the same way. 1 = same direction, 0 = unrelated | The angle between two arrows |
| **Chunk** | The piece of a document we embed — here one `##` section of a help article | One FAQ answer |
| **Semantic search** | Embed the question, compare with every chunk, return the closest | "Find answers that *mean* this" |
| **Top-k** | The k best matches (we use k = 3) | The top 3 search results |
| **hit@3** | Of our test questions, how many had the right article somewhere in the top 3 | "Was it on the first page?" |
| **MRR** | Mean reciprocal rank: 1.0 if the right answer is always 1st, 0.5 if usually 2nd | "How high up was it?" |

### How today is built
```
OFFLINE (once):  14 help articles ─► split into 53 sections ─► embedding model ─► 53 vectors (stored)
ONLINE (every question):  question ─► same model ─► 1 vector ─► cosine with all 53 ─► top 3 sections
                                                                                       └─► (Module 5: give them to the LLM)
```
**Golden rule:** the documents and the questions **must** go through the same model. Switch models → re-embed everything.

On the course board, this is the new **Retrieval** box next to FastAPI (S23) and the Gateway (S23b).

---
# Part B — Run it
### B1. Load the help centre
The 14 articles are in `module04/help_centre/` — the same Northwind Store as MP1. Open one in VS Code: front matter, a title,
then `##` sections. **One section = one chunk**, because one section answers one kind of question, and its heading makes a
ready-made citation ("Warranty > Warranty periods").

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "module04") if (Path.cwd() / "module04").exists() else str(Path.cwd()))

import numpy as np
import s24_utils as U
from s24_queries import CASES, OUT_OF_SCOPE, PASS_MARK

chunks = U.load_chunks()
print(len({c.doc_id for c in chunks}), "articles,", len(chunks), "sections")
c = chunks[0]
print("\nexample chunk:")
print("  source :", c.source)
print("  text   :", c.text[:160], "...")

### B2. The old way — keyword search
Score = how many of the question's words appear in the section (ignoring small words like *the*, *my*). Watch what happens.

In [ ]:
for q in ["my earbuds stopped working", "the parcel arrived smashed", "swap my sneakers for a bigger size"]:
    print(f"\n{q!r}")
    res = U.keyword_search(q, chunks)
    if not res:
        print("   nothing - not one word in common")
    for score, c in res:
        print(f"   {score:.0f} shared words   {c.source}")

"Stopped **working**" matched *Coupon not working* — the right word, the wrong meaning. "Smashed" found nothing at all.
Now measure it on all ten test questions (`module04/s24_queries.py`):

In [ ]:
kw = U.evaluate(lambda q, k: U.keyword_search(q, chunks, k), CASES)
print(f"keyword search: hit@3 {kw['hits']}/{kw['n']}   MRR {kw['mrr']:.2f}   (pass mark {PASS_MARK})")
for row in kw["rows"]:
    print("  ", "ok  " if row["rank"] else "MISS", row["question"])

### B3. Your first embedding
One sentence in, 384 numbers out. **No single number means "damaged" or "refund"** — the meaning is in the *pattern*
across all 384, the way a face isn't one pixel.

In [ ]:
emb = U.Embedder("bge-small")            # first run downloads the model (~67 MB), then it's cached
v = emb.embed(["return a damaged item"])[0]
print("model     :", emb.model_id)
print("dimensions:", len(v))
print("first 8   :", np.round(v[:8], 3))
print("length    :", round(float(np.linalg.norm(v)), 3), "(we scale every vector to length 1)")

### B4. Cosine similarity — by hand first, then for real
Think of two arrows from the same point. **Cosine** measures the angle between them:

| Arrows | Angle | Cosine |
|---|---|---|
| Same direction | 0° | 1.0 |
| At right angles | 90° | 0.0 |
| Opposite | 180° | −1.0 |

Formula: `cos = (a · b) / (|a| × |b|)`. Length doesn't matter — only direction. A long article and a short question can still
"point the same way".

In [ ]:
print(U.cosine([1, 0], [1, 0]), U.cosine([1, 0], [0, 1]), U.cosine([1, 0], [-1, 0]), U.cosine([3, 4], [6, 8]))

texts = ["the parcel arrived smashed", "my item came damaged", "the product was broken", "gift card balance"]
vecs = emb.embed(texts)
print()
for i, a in enumerate(texts):
    print(f"{a:30}", "  ".join(f"{U.cosine(vecs[i], vecs[j]):.2f}" for j in range(len(texts))))

**Try it:** add a sentence of your own (in Telugu or Hindi too — bge-small is English-first, so watch the score drop).
Real-world note: with modern models even unrelated sentences rarely score near 0; what matters is the **ranking**,
not the absolute number.

### B5. Meaning search
Embed all 53 sections once (the **offline** half), then embed each question and rank by cosine (the **online** half).
The vectors are cached in `module04/.s24_cache/` so the second run is instant.

In [ ]:
index = U.Index(emb)
print(f"index: {index.vectors.shape[0]} vectors x {index.vectors.shape[1]} numbers"
      + (f", built in {index.seconds:.1f} s" if index.seconds else " (from cache)"))
for q in ["my earbuds stopped working", "the parcel arrived smashed"]:
    print(f"\n{q!r}")
    for score, c in index.search(q):
        print(f"   {score:.3f}   {c.source}")

### B6. Measure it — never trust a demo
Same ten questions as keyword search. **hit@3** asks "was the right article in the top 3?" **MRR** asks "how high up?"

In [ ]:
se = U.evaluate(index.search, CASES)
print(f"keyword : hit@3 {kw['hits']}/{kw['n']}   MRR {kw['mrr']:.2f}")
print(f"meaning : hit@3 {se['hits']}/{se['n']}   MRR {se['mrr']:.2f}   (pass mark {PASS_MARK})")
for row in se["rows"]:
    print("  ", f"#{row['rank']}" if row["rank"] else "MISS", row["question"], "->", row["found"])

print("\nquestions the help centre CAN'T answer - search still returns something:")
for q in OUT_OF_SCOPE:
    score, c = index.search(q, 1)[0]
    print(f"   best {score:.2f}  {c.source:45}  <- {q!r}")

Look at the out-of-scope scores: **search always returns the nearest thing, even when nothing is relevant.** If we gave
that to an LLM, it would happily answer "do you sell used cars?" from the Shipping article. The fix is a **relevance
threshold** — ShopAssist Pro uses 0.63 for bge-small, tuned on exactly this kind of measurement (Module 5).

### B7. Compare two models
Choosing a model is an engineering decision: quality, dimensions (storage and speed), size, language, cost.
`all-MiniLM-L6-v2` is the classic small model; `bge-small-en-v1.5` is newer and trained for retrieval.

In [ ]:
rows = []
for name in ["bge-small", "minilm"]:
    idx = U.Index(U.Embedder(name), chunks)
    r = U.evaluate(idx.search, CASES)
    rows.append((idx.embedder.model_id, idx.embedder.dims, r["hits"], r["mrr"]))
print(f"{'model':42}{'dims':>6}{'hit@3':>8}{'MRR':>7}")
for m, d, h, mrr in rows:
    print(f"{m:42}{d:>6}{h:>6}/10{mrr:>7.2f}")

**Try it (optional):** if Ollama runs on your laptop with `nomic-embed-text` pulled (Session 9), add `"nomic"` to the list —
768 dimensions. Is twice the size twice as good?

**Golden rule, again:** you can't search bge vectors with a MiniLM question. Different models = different maps.

---
# Part C — Your homework: `s24/search.py`
You'll write the core of semantic search yourself, in **your own work repo**:

| TODO | What | Lines |
|---|---|---|
| 1 | `cosine(a, b)` with numpy | 3 |
| 2 | `top_k(query, matrix, k)` — the k most similar rows, best first | 2–4 |
| 3 | `split_sections(markdown)` — one article → (heading, text) per `##` section | 6–8 |
| 4 | `MY_CASES` — three customer questions of your own, each with the article that answers it | 3 |

**Steps** (course repo folder, PowerShell):
```powershell
uv run python module04/s24_setup.py                 # copies the file to <your work repo>\s24\search.py
# edit it in VS Code: search for TODO
uv run python "<your work repo>\s24\search.py"      # runs YOUR functions on the ten questions + yours; writes results.md
uv run pytest tests/test_s24.py -rs                 # the checker (offline)
cd "<your work repo>" ; git add s24 ; git commit -m "S24: semantic search" ; git push
```
`s24_setup.py` reads `WORK_REPO` from your `.env`, like Session 23. Each TODO's docstring has a **Check:** line you can paste
into this notebook to test it before running the whole file.

**Stretch (not graded):** change `top_k` to drop any result below a score threshold and pick the threshold from the
out-of-scope numbers in B6.

---
# Part D — Interview questions

| Question | Fresher answer | Experienced answer |
|---|---|---|
| What is an embedding? | A list of numbers that represents the meaning of a text, so similar texts get similar numbers. | A dense vector from a model trained (often contrastively) so semantically similar inputs land close together; used for retrieval, clustering, deduplication and classification. |
| Why cosine similarity and not plain distance? | Cosine looks at direction, not length, so a long article and a short question can still match. | With normalised vectors cosine and dot product rank identically and dot product is cheaper; Euclidean on unnormalised vectors mixes length into similarity. |
| How do you choose an embedding model? | Check a leaderboard like MTEB and test it on my own questions. | Retrieval score on *my* golden set (not just MTEB), dimensions vs storage/latency, max input length, language coverage, licence, hosted vs self-hosted cost, and whether it needs query/passage prefixes. |
| What happens if you change the embedding model? | You must re-embed all the documents. | Full re-index, ideally side-by-side (shadow index) with an eval gate before switching traffic; vectors from different models are not comparable. |
| Why doesn't search say "nothing found"? | It always returns the nearest match. | Nearest-neighbour search has no notion of "irrelevant"; you add a similarity threshold calibrated on in-scope vs out-of-scope questions, and test it in CI. |
| Dense vs sparse? | Dense = meaning, sparse = exact words. | Sparse (BM25) wins on exact terms, codes and names; dense on paraphrase. Production systems combine both — hybrid search with rank fusion (S27). |

### What's next
**S25 — Vector databases.** Today we compared a question with all 53 vectors. That's fine for 53 and hopeless for 50 million.
Next session: how vector databases find the nearest vectors fast (HNSW), store metadata and filter by it.